In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"lfreedom2750","key":"fb46b035b65134128288a6ce5f370912"}'}

In [ ]:
!pip install -q kaggle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d lfreedom2750/fakeface-train-data-v2
!unzip -q fakeface-train-data-v2.zip -d train_dataset

Dataset URL: https://www.kaggle.com/datasets/lfreedom2750/fakeface-train-data-v2
License(s): unknown
 97% 2.15G/2.21G [00:04<00:00, 471MB/s]
100% 2.21G/2.21G [00:04<00:00, 488MB/s]


In [ ]:
!kaggle datasets download -d lfreedom2750/fakeface-valid-data-v2
!unzip -q fakeface-valid-data-v2.zip -d valid_dataset

Dataset URL: https://www.kaggle.com/datasets/lfreedom2750/fakeface-valid-data-v2
License(s): unknown
 87% 646M/745M [00:00<00:00, 1.32GB/s]
100% 745M/745M [00:00<00:00, 1.28GB/s]


In [ ]:
!kaggle datasets download -d lfreedom2750/fakeface-test-data-v2
!unzip -q fakeface-test-data-v2.zip -d test_dataset

Dataset URL: https://www.kaggle.com/datasets/lfreedom2750/fakeface-test-data-v2
License(s): unknown
 87% 649M/747M [00:00<00:00, 1.25GB/s]
100% 747M/747M [00:00<00:00, 1.08GB/s]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import csv
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from PIL import Image

In [ ]:
from torchvision import datasets
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder("/content/train_dataset", transform=transform)
val_dataset   = datasets.ImageFolder("/content/valid_dataset", transform=transform)
test_dataset = datasets.ImageFolder("/content/test_dataset", transform=transform)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
test_loader   = DataLoader(test_dataset, batch_size=256, shuffle=False,num_workers=4, pin_memory=True)

In [ ]:
from transformers import MobileNetV1ForImageClassification, MobileNetV1Config
import torch
import torch.nn as nn

class MobileNetV1(nn.Module):
    def __init__(self):
        super().__init__()
        # Load pretrained MobileNetV1 model
        self.model = MobileNetV1ForImageClassification.from_pretrained(
            "google/mobilenet_v1_1.0_224"
        )
        # Modify the classifier for 2 classes
        num_features = self.model.classifier.in_features
        self.model.classifier = nn.Linear(num_features, 2)

    def forward(self, x):
        return self.model(x).logits

In [ ]:
import time
import torch
import torch.nn as nn
from torchvision import models
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = MobileNetV1()
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

epoch_times = []
start_training = time.time()

for epoch in range(10):
    start_epoch = time.time()

    model.train()
    total_loss = 0
    total_batches = len(train_loader)

    for x, y in tqdm(train_loader, desc=f"[Epoch {epoch+1}] Training", leave=False):
        x, y = x.to(device), y.to(device)
        loss = criterion(model(x), y)
        print(f"Loss: {loss.item():.4f}")
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    avg = total_loss / len(train_loader)

    model.eval()
    correct, total = 0, 0
    all_labels, all_probs = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            probs = torch.softmax(outputs, dim=1)[:, 1]

            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    val_acc = correct / total
    val_auc = roc_auc_score(all_labels, all_probs)

    epoch_time = time.time() - start_epoch
    epoch_times.append(epoch_time)

    print(f"[MobileNetV3] Epoch {epoch+1} | Train Loss: {avg:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f} | Time: {epoch_time:.2f}s")

total_time = time.time() - start_training
avg_time = sum(epoch_times) / len(epoch_times)

print(f"\nTotal training time: {total_time:.2f}s")
print(f"Average time per epoch: {avg_time:.2f}s")

torch.save(model.state_dict(), "mobilenetv3.pth")

cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/69.6k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

[Epoch 1] Training:   0%|          | 0/473 [00:00<?, ?it/s]

...

[Epoch 10] Training: 100%|█████████▉| 471/473 [04:41<00:01,  1.68it/s]

Loss: 0.0040


[Epoch 10] Training: 100%|██████████| 473/473 [04:42<00:00,  2.01it/s]

Loss: 0.0006


[MobileNetV3] Epoch 10 | Train Loss: 0.0018 | Val Acc: 0.9649 | Val AUC: 0.9955 | Time: 317.42s

Total training time: 3175.60s
Average time per epoch: 317.56s


In [ ]:
from sklearn.metrics import classification_report

def evaluate_model(model, dataloader, device='cuda'):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.cpu().numpy()
            outputs = model(x)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y)

    print("\n=== Classification Report ===")
    print(classification_report(all_labels, all_preds, digits=4))

In [ ]:
evaluate_model(model, test_loader)


=== Classification Report ===
              precision    recall  f1-score   support

           0     0.9542    0.9780    0.9660     20000
           1     0.9775    0.9530    0.9651     20000

    accuracy                         0.9656     40000
   macro avg     0.9658    0.9655    0.9655     40000
weighted avg     0.9658    0.9656    0.9655     40000



In [ ]:
torch.save(model.state_dict(), "/content/mobilenetv1.pth")